# Binary Classification: `human` vs `macro`

- 입력: trial JSON 파일(1 trial = 1 file) 또는 DB export JSONL
- 입력 feature: `services/ai/configs/feature_config.yaml`
- 전처리: 결측률 10% 초과 feature 제외
- 모델: 여러 모델을 쉽게 교체/비교
- 앙상블: 간단한 soft-voting
- Cross test: 다른 데이터 경로로 추가 평가


## 0) 환경 준비

필요 패키지가 없으면 한 번만 설치:

```bash
pip install pandas scikit-learn pyyaml xgboost
```


In [10]:
from __future__ import annotations

import json
# from dataclasses import dataclass  # (현재 노트북에서는 미사용)
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import yaml

# (선택) XGBoost - 설치가 안 되어 있으면 자동으로 스킵하도록 처리합니다.
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

from sklearn.base import BaseEstimator
# from sklearn.compose import ColumnTransformer  # (현재 노트북에서는 미사용)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 1) 데이터 경로 설정

`DATA_DIR`에 trial JSON 파일들이 들어있는 폴더 경로를 넣어주세요.
- 허용: `trial_*.json` (우선) 또는 일반 `*.json`
- 각 JSON에는 `label`(또는 `summary.label`)과 `metrics`/`features`가 포함되어야 합니다.


In [ ]:
# ==========================
# 실험 설정(여기만 수정)
# ==========================

# 1) 데이터 경로: trial JSON 파일들이 들어있는 폴더
# - 이 노트북은 폴더 내의 `trial_*.json`을 우선적으로 읽고, 없으면 `*.json`을 읽습니다.
DATA_DIR = r"C:\Users\SSAFY\Desktop\ws\free\develop-ai\S14P31A203\services\ai\data\v1_v2"

# 2) (선택) cross test 경로: 다른 데이터셋으로 최종 모델 성능 확인
CROSS_TEST_DIR = r"C:\Users\SSAFY\Desktop\ws\free\develop-ai\S14P31A203\services\ai\data\data_c1"

# 3) 라벨 설정
# - y=1(양성): macro, y=0(음성): human
# - 데이터 JSON의 label은 `record['label']` 우선, 없으면 `record['summary']['label']`에서 찾습니다.
LABEL_KEY = "label"
POSITIVE_LABEL = "macro"
NEGATIVE_LABEL = "human"

# 3-1) 라벨 매핑(혼동 방지)
# - 데이터가 ALLOW/BLOCK으로 들어와도 학습에서는 human/macro로 통일해서 쓰기 위함
# - 예: BLOCK -> macro, ALLOW -> human
# - 시각화/리포트에서는 최종 라벨이 macro/human으로 보이게 됩니다.
LABEL_MAP = {
    "BLOCK": "macro",
    "macro": "macro",
    "ALLOW": "human",
    "human": "human",
}

# 4) feature 설정
# - `services/ai/configs/feature_config.yaml`에 여러 feature list가 정의되어 있음
# - 아래 KEY를 바꾸면 다른 feature 셋으로 실험 가능
FEATURE_CONFIG_PATH = Path(r"C:\Users\SSAFY\Desktop\ws\free\develop-ai\S14P31A203\services\ai\configs\feature_config.yaml")
FEATURE_LIST_KEY = "input_features_chan"

# 5) 전처리 설정
# - 결측률이 이 값(기본 10%)을 초과하는 feature는 제외
MISSING_DROP_THRESHOLD = 0.10

# 6) 학습/평가 설정
RANDOM_STATE = 42
TOP_K_ENSEMBLE = 3

# 7) 모델 후보
# - xgb는 xgboost가 설치되어 있을 때만 사용됨
MODEL_CANDIDATES = ["logreg", "rf", "hgb", "gb", "xgb"]


## 2) feature config 로드 (`feature_config.yaml`)

기본은 `input_features_chan`을 사용합니다.
필요하면 YAML 내 다른 feature list로 바꿔서 실험할 수 있습니다.


In [12]:
# path-config 셀에서 설정한 FEATURE_CONFIG_PATH / FEATURE_LIST_KEY 사용
cfg = yaml.safe_load(FEATURE_CONFIG_PATH.read_text(encoding="utf-8"))
candidate_features: List[str] = list(cfg.get(FEATURE_LIST_KEY, []))

print("Feature list key:", FEATURE_LIST_KEY)
print("Candidate features:", len(candidate_features))
candidate_features[:10]

AttributeError: 'str' object has no attribute 'read_text'

## 3) 데이터 로드 (JSON trials) → DataFrame

label 추출 규칙:
- 우선 `record[label_key]`
- 없으면 `record['summary']['label']`

feature(metrics) 추출 규칙:
- 우선 `record['metrics']`
- 없으면 `record['features']`


In [ ]:
JsonRecord = Dict[str, Any]


def resolve_label(record: JsonRecord, *, label_key: str = "label") -> str:
    """label을 추출하고(LABEL_KEY → summary.label), LABEL_MAP으로 표준화합니다."""
    raw = record.get(label_key)
    if raw is None and label_key == "label":
        summary = record.get("summary")
        if isinstance(summary, dict):
            raw = summary.get("label")
    label = str(raw).strip().lower() if raw is not None else ""

    # config 셀에서 정의한 LABEL_MAP 적용 (ex: block->macro, allow->human)
    label_map = globals().get("LABEL_MAP") or {}
    if isinstance(label_map, dict):
        label = str(label_map.get(label, label)).strip().lower()
    return label


def resolve_metrics(record: JsonRecord) -> Dict[str, Any]:
    """Return feature dict from common shapes."""
    metrics = record.get("metrics")
    if isinstance(metrics, dict):
        return metrics
    features = record.get("features")
    if isinstance(features, dict):
        return features
    return {}


def iter_trial_json_paths(data_dir: str | Path, pattern: str = "*.json") -> List[Path]:
    root = Path(data_dir).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"DATA_DIR not found: {root}")
    if root.is_file() and root.suffix.lower() == ".json":
        return [root]
    if root.is_dir():
        # Prefer trial_*.json if present; otherwise take all *.json
        trial_paths = sorted(root.glob("trial_*.json"))
        if trial_paths:
            return trial_paths
        return sorted(root.glob(pattern))
    raise ValueError(f"Unsupported path type: {root}")


def load_trials_df(
    data_dir: str | Path,
    *,
    label_key: str = "label",
    positive_label: str = POSITIVE_LABEL,
    negative_label: str = NEGATIVE_LABEL,
    features: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, pd.Series]:
    """Load trial JSON files into X (DataFrame) and y (Series)."""
    paths = iter_trial_json_paths(data_dir)
    rows: List[Dict[str, Any]] = []
    y: List[int] = []

    for path in paths:
        record = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(record, dict):
            continue

        label = resolve_label(record, label_key=label_key)
        if label not in {positive_label, negative_label}:
            # 이 노트북은 이진분류(human/macro) 목적이므로 다른 라벨(allow/block 등)은 제외
            continue

        metrics = resolve_metrics(record)
        row: Dict[str, Any] = {}
        use_features = features if features is not None else list(metrics.keys())
        for key in use_features:
            row[key] = metrics.get(key)

        rows.append(row)
        y.append(1 if label == positive_label else 0)

    X = pd.DataFrame(rows)
    y_s = pd.Series(y, name="label")
    return X, y_s


X_raw, y = load_trials_df(DATA_DIR, label_key=LABEL_KEY, features=candidate_features)
print("Loaded X:", X_raw.shape, "y:", y.shape)
print(y.value_counts(dropna=False))
X_raw.head()

## 4) 전처리

요구사항:
- **결측률 10% 초과** feature는 제외
- 결측률 테이블 + 제외된 feature 리스트를 셀 출력으로 기록

`pd.to_numeric(errors='coerce')`로 숫자 변환을 시도하며 실패는 NaN으로 처리합니다.


In [ ]:
# 타입 변환: bool/문자 bool은 0/1로, 나머지는 숫자 변환 시도(실패는 NaN)
X = X_raw.copy()
for col in X.columns:
    if X[col].dtype == object:
        # Convert common bool strings safely
        X[col] = X[col].replace({"true": 1, "false": 0, True: 1, False: 0})
    X[col] = pd.to_numeric(X[col], errors="coerce")

missing_rate = X.isna().mean().sort_values(ascending=False)
missing_table = pd.DataFrame({"missing_rate": missing_rate, "missing_pct": (missing_rate * 100).round(2)})

DROP_THRESHOLD = float(MISSING_DROP_THRESHOLD)
drop_features = missing_rate[missing_rate > DROP_THRESHOLD].index.tolist()
keep_features = [f for f in X.columns.tolist() if f not in drop_features]

print(f"Total candidate features: {len(X.columns)}")
print(f"Drop threshold: {DROP_THRESHOLD:.0%}")
print(f"Dropped features: {len(drop_features)}")
print(f"Kept features: {len(keep_features)}")

display(missing_table.head(30))
drop_features

In [ ]:
X = X[keep_features]
print("Final X shape:", X.shape)

# keep_features가 비어있으면(=결측률 기준으로 전부 drop) 이후 학습이 불가능합니다.
# 이 경우는 보통 아래 원인 중 하나입니다.
# - MISSING_DROP_THRESHOLD가 너무 낮음 (예: 0.10이 너무 빡셈)
# - FEATURE_LIST_KEY가 데이터에 없는 feature 리스트를 가리킴
# - 데이터의 metrics/features 키가 비어있거나 다른 구조
if X.shape[1] == 0:
    print("\n[ERROR] 전처리 이후 남은 feature가 0개입니다.")
    print("- missing_table 상위(결측률 높은 순)를 확인하세요.")
    print("- 해결 방법: (1) MISSING_DROP_THRESHOLD를 0.3~0.5로 올리기, (2) FEATURE_LIST_KEY 변경, (3) 데이터 구조 확인")
    display(missing_table.head(50))
else:
    display(X.describe().T.head(10))


## 5) Train/Validation/Test 분할

- Train: 70%
- Val: 15%
- Test: 15%

라벨 비율을 유지하기 위해 stratify로 분할합니다.


In [ ]:
# path-config 셀에서 설정한 RANDOM_STATE 사용

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp
)

print("train:", X_train.shape, y_train.value_counts().to_dict())
print("val  :", X_val.shape, y_val.value_counts().to_dict())
print("test :", X_test.shape, y_test.value_counts().to_dict())

## 6) 모델 목록 (빠르게 교체/비교)

Pipeline 구성:
- `SimpleImputer(median)` for missing values
- Optional `StandardScaler` for linear models
- Classifier


In [ ]:
def make_model(model_name: str, *, random_state: int = 42) -> BaseEstimator:
    """Return an untrained sklearn estimator."""
    model_name = model_name.lower().strip()

    if model_name == "logreg":
        return LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=random_state,
        )
    if model_name == "rf":
        return RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        )
    if model_name == "gb":
        return GradientBoostingClassifier(random_state=random_state)
    if model_name == "hgb":
        return HistGradientBoostingClassifier(random_state=random_state)
    if model_name == "xgb":
        # xgboost가 설치되어 있어야 합니다. (imports 셀의 XGBClassifier 참고)
        if XGBClassifier is None:
            raise ValueError("XGBClassifier is not available. Install xgboost or remove 'xgb' from MODEL_CANDIDATES.")
        return XGBClassifier(
            n_estimators=600,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1,
            eval_metric="logloss",
        )

    raise ValueError(f"Unknown model_name: {model_name}")


def needs_scaling(model_name: str) -> bool:
    return model_name.lower().strip() in {"logreg"}


def make_pipeline(model_name: str, *, random_state: int = 42) -> Pipeline:
    steps = []
    steps.append(("imputer", SimpleImputer(strategy="median")))
    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", make_model(model_name, random_state=random_state)))
    return Pipeline(steps)


# NOTE: 모델 후보는 path-config 셀에서 한 번에 설정합니다.
MODEL_CANDIDATES

In [ ]:
def predict_proba_safe(model: BaseEstimator, X: pd.DataFrame) -> np.ndarray:
    """Return positive-class probability. Calibrate if needed."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    # Fallback: calibrate using internal CV.
    calibrated = CalibratedClassifierCV(model, method="sigmoid", cv=3)
    calibrated.fit(X_train, y_train)
    return calibrated.predict_proba(X)[:, 1]


def evaluate_binary(model: BaseEstimator, X: pd.DataFrame, y_true: pd.Series, *, title: str) -> Dict[str, float]:
    y_pred = model.predict(X)
    proba = predict_proba_safe(model, X)
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, proba)) if len(set(y_true)) > 1 else float("nan"),
    }

    print("=" * 80)
    print(title)
    print(json.dumps(out, indent=2))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nClassification report:\n", classification_report(y_true, y_pred, digits=4))
    return out

## 7) 모델 학습 & 검증 비교

아래 셀은 후보 모델을 모두 학습하고, validation F1 기준으로 정렬합니다.


In [ ]:
results = []
trained: Dict[str, Pipeline] = {}

for name in MODEL_CANDIDATES:
    pipe = make_pipeline(name, random_state=RANDOM_STATE)
    pipe.fit(X_train, y_train)
    trained[name] = pipe
    metrics = evaluate_binary(pipe, X_val, y_val, title=f"[VAL] model={name}")
    results.append({"model": name, **metrics})

df_results = pd.DataFrame(results).sort_values(by="f1", ascending=False)
df_results

## 8) best 단일 모델 테스트

validation 성능이 가장 좋은 모델로 test set을 평가합니다.


In [ ]:
best_model_name = df_results.iloc[0]["model"]
best_model = trained[str(best_model_name)]

evaluate_binary(best_model, X_test, y_test, title=f"[TEST] best single model={best_model_name}")

## 9) 앙상블 (soft voting)

- validation 상위 K개 모델 선택
- `VotingClassifier(voting='soft')`로 앙상블 학습
- test set 평가


In [ ]:
# path-config 셀에서 설정한 TOP_K_ENSEMBLE 사용
TOP_K = int(TOP_K_ENSEMBLE)
top_models = df_results.head(TOP_K)["model"].tolist()
print("Top models:", top_models)

# VotingClassifier는 (name, estimator) 튜플을 받습니다. (pipeline도 estimator로 OK)
estimators = [(name, trained[name]) for name in top_models]

ensemble = VotingClassifier(estimators=estimators, voting="soft")

# 최종 평가를 위해 train+val 합쳐서 앙상블 학습
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

ensemble.fit(X_trainval, y_trainval)
evaluate_binary(ensemble, X_test, y_test, title=f"[TEST] soft-voting ensemble top_k={TOP_K}")

## 10) Cross test (다른 데이터로 추가 평가)

`CROSS_TEST_DIR`에 다른 데이터 디렉토리(trial JSON 파일들)를 넣으면 cross test를 수행합니다.
학습 때 선택된 `keep_features`를 그대로 사용하고, 위에서 학습한 앙상블을 평가합니다.


In [ ]:
def load_cross_test(data_dir: str | Path) -> Tuple[pd.DataFrame, pd.Series]:
    X_raw2, y2 = load_trials_df(data_dir, features=candidate_features)
    X2 = X_raw2.copy()
    for col in X2.columns:
        if X2[col].dtype == object:
            X2[col] = X2[col].replace({"true": 1, "false": 0, True: 1, False: 0})
        X2[col] = pd.to_numeric(X2[col], errors="coerce")
    X2 = X2[keep_features]
    return X2, y2


if CROSS_TEST_DIR:
    X_cross, y_cross = load_cross_test(CROSS_TEST_DIR)
    print("Cross-test shape:", X_cross.shape, y_cross.shape)
    evaluate_binary(ensemble, X_cross, y_cross, title=f"[CROSS TEST] {CROSS_TEST_DIR}")
else:
    print("CROSS_TEST_DIR가 None 입니다. cross test를 하려면 경로를 지정하세요.")